# Actividad 4 - Optimización de un modelo de ensamble

**Clase:** Fundamentos Algoritmos de Aprendizaje Automático  
**Dataset:** Breast Cancer Wisconsin (Diagnostic), disponible en `sklearn.datasets.load_breast_cancer`  
**Fuente:** UCI Machine Learning Repository / Scikit-learn  
**Problema:** clasificación binaria de tumores de mama  
**Variable objetivo:** `target` (0 = malignant, 1 = benign)  
**Cada observación:** una muestra con 30 mediciones numéricas de núcleos celulares.

**Métrica principal:** F1-score. Se usa porque combina precision y recall y permite evaluar mejor el equilibrio entre errores de clasificación que accuracy por sí sola.  
**Métrica complementaria:** ROC-AUC.

## 1. Carga y preparación de datos

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectFromModel
from sklearn.pipeline import Pipeline
from sklearn.metrics import f1_score, roc_auc_score, confusion_matrix
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from time import perf_counter

random_state = 42

# load the public dataset
data = load_breast_cancer(as_frame=True)
df = data.frame.copy()

print("shape:", df.shape)
print("missing values:", int(df.isna().sum().sum()))
print("duplicates:", int(df.duplicated().sum()))
print("\ndata types:")
display(df.dtypes.value_counts())
print("\ntarget distribution:")
display(df["target"].value_counts().rename(index={0:"malignant", 1:"benign"}).to_frame("count"))

# separate predictors and target
x = df.drop(columns="target")
y = df["target"]

# fixed 80/20 split with stratification
x_train, x_test, y_train, y_test = train_test_split(
    x, y,
    test_size=0.20,
    stratify=y,
    random_state=random_state
)

print("train rows:", len(x_train))
print("test rows:", len(x_test))
print("features:", x_train.shape[1])

# save a reproducible copy of the dataset
df.to_csv("breast_cancer_wisconsin.csv", index=False)

Todas las variables predictoras son numéricas, no hay valores faltantes y Random Forest no requiere escalamiento. Se utiliza estratificación para conservar la proporción de clases.

## 2. Modelo base

In [ ]:
# baseline random forest
base_model = RandomForestClassifier(
    n_estimators=100,
    random_state=random_state,
    n_jobs=-1
)

start = perf_counter()
base_model.fit(x_train, y_train)
base_time = perf_counter() - start

base_pred = base_model.predict(x_test)
base_prob = base_model.predict_proba(x_test)[:, 1]

base_f1 = f1_score(y_test, base_pred)
base_auc = roc_auc_score(y_test, base_prob)

print(f"f1-score: {base_f1:.4f}")
print(f"roc-auc: {base_auc:.4f}")
print("features:", x_train.shape[1])
print(f"time: {base_time:.4f} seconds")

## 3. Selección de características con SelectFromModel

In [ ]:
# fit feature selection only on training data to avoid leakage
selector = SelectFromModel(
    RandomForestClassifier(
        n_estimators=200,
        random_state=random_state,
        n_jobs=-1
    ),
    threshold="median"
)

selector.fit(x_train, y_train)

selected_features = x_train.columns[selector.get_support()].tolist()

print("original features:", x_train.shape[1])
print("selected features:", len(selected_features))
print("\nfeatures kept:")
for feature in selected_features:
    print("-", feature)

`SelectFromModel` usa la importancia de variables de un Random Forest auxiliar. Con `threshold="median"` conserva las variables cuya importancia es al menos la mediana. La selección se ajusta únicamente con entrenamiento, evitando fuga de información.

In [ ]:
# transform train and test using the fitted selector
x_train_reduced = selector.transform(x_train)
x_test_reduced = selector.transform(x_test)

reduced_model = RandomForestClassifier(
    n_estimators=100,
    random_state=random_state,
    n_jobs=-1
)

start = perf_counter()
reduced_model.fit(x_train_reduced, y_train)
reduced_time = perf_counter() - start

reduced_pred = reduced_model.predict(x_test_reduced)
reduced_prob = reduced_model.predict_proba(x_test_reduced)[:, 1]

reduced_f1 = f1_score(y_test, reduced_pred)
reduced_auc = roc_auc_score(y_test, reduced_prob)

reduction_table = pd.DataFrame([
    {
        "configuration": "base model",
        "features": x_train.shape[1],
        "main_metric_f1": base_f1,
        "complementary_roc_auc": base_auc,
        "training_time_seconds": base_time
    },
    {
        "configuration": "feature selection",
        "features": len(selected_features),
        "main_metric_f1": reduced_f1,
        "complementary_roc_auc": reduced_auc,
        "training_time_seconds": reduced_time
    }
])

display(reduction_table.round(4))

La tabla permite comprobar si la reducción mejora el desempeño, conserva resultados semejantes con menos variables o elimina demasiada información.

## 4. Ajuste de hiperparámetros con GridSearchCV

In [ ]:
# pipeline keeps feature selection inside cross-validation
optimization_pipeline = Pipeline([
    (
        "selector",
        SelectFromModel(
            RandomForestClassifier(
                n_estimators=200,
                random_state=random_state,
                n_jobs=-1
            ),
            threshold="median"
        )
    ),
    (
        "model",
        RandomForestClassifier(
            random_state=random_state,
            n_jobs=-1
        )
    )
])

param_grid = {
    "model__n_estimators": [100, 200],
    "model__max_depth": [None, 8, 16],
    "model__min_samples_split": [2, 5],
    "model__max_features": ["sqrt", 0.7]
}

cv_strategy = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=random_state
)

grid_search = GridSearchCV(
    optimization_pipeline,
    param_grid=param_grid,
    scoring="f1",
    cv=cv_strategy,
    n_jobs=-1,
    refit=True
)

start = perf_counter()
grid_search.fit(x_train, y_train)
optimization_time = perf_counter() - start

print("best cv f1:", round(grid_search.best_score_, 4))
print("best parameters:")
print(grid_search.best_params_)
print(f"grid search time: {optimization_time:.4f} seconds")

In [ ]:
hyperparameter_table = pd.DataFrame([
    {
        "hyperparameter": "n_estimators",
        "values_explored": "[100, 200]",
        "best_value": grid_search.best_params_["model__n_estimators"]
    },
    {
        "hyperparameter": "max_depth",
        "values_explored": "[None, 8, 16]",
        "best_value": grid_search.best_params_["model__max_depth"]
    },
    {
        "hyperparameter": "min_samples_split",
        "values_explored": "[2, 5]",
        "best_value": grid_search.best_params_["model__min_samples_split"]
    },
    {
        "hyperparameter": "max_features",
        "values_explored": "['sqrt', 0.7]",
        "best_value": grid_search.best_params_["model__max_features"]
    }
])

display(hyperparameter_table)

Se usa validación cruzada estratificada de 5 particiones y F1-score como criterio. La validación cruzada compara configuraciones en varias particiones y por eso es más estable que depender de una sola división. El conjunto de prueba se reserva para la evaluación final y no se utiliza para escoger hiperparámetros.

## 5. Evaluación final de los tres modelos

In [ ]:
best_model = grid_search.best_estimator_

optimized_pred = best_model.predict(x_test)
optimized_prob = best_model.predict_proba(x_test)[:, 1]

optimized_f1 = f1_score(y_test, optimized_pred)
optimized_auc = roc_auc_score(y_test, optimized_prob)

best_selector = best_model.named_steps["selector"]
optimized_features = int(best_selector.get_support().sum())

results = pd.DataFrame([
    {
        "model": "base",
        "f1_score": base_f1,
        "roc_auc": base_auc,
        "features": x_train.shape[1],
        "time_seconds": base_time
    },
    {
        "model": "reduced",
        "f1_score": reduced_f1,
        "roc_auc": reduced_auc,
        "features": len(selected_features),
        "time_seconds": reduced_time
    },
    {
        "model": "optimized",
        "f1_score": optimized_f1,
        "roc_auc": optimized_auc,
        "features": optimized_features,
        "time_seconds": optimization_time
    }
])

display(results.round(4))

## 6. Visualización comparativa

In [ ]:
plot_data = results.set_index("model")[["f1_score", "roc_auc"]]

ax = plot_data.plot(kind="bar", figsize=(8, 5))
ax.set_title("comparison of model performance")
ax.set_ylabel("score")
ax.set_ylim(0, 1.05)
ax.set_xlabel("model")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# confusion matrix for the optimized model
cm = confusion_matrix(y_test, optimized_pred)

fig, ax = plt.subplots(figsize=(5, 4))
ax.imshow(cm)
ax.set_title("optimized model confusion matrix")
ax.set_xlabel("predicted class")
ax.set_ylabel("true class")
ax.set_xticks([0, 1])
ax.set_yticks([0, 1])

for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, cm[i, j], ha="center", va="center")

plt.tight_layout()
plt.show()

## 7. Análisis
- **Interpretabilidad:** el modelo reducido usa menos variables y conserva nombres originales.
- **Escalamiento:** Random Forest no es sensible a que las variables tengan escalas distintas.
- **Complejidad:** GridSearchCV aumenta considerablemente el tiempo porque entrena muchas configuraciones.
- **Estabilidad:** se utiliza validación cruzada estratificada de 5 folds.
- **Sobreajuste:** reservar el test y limitar el espacio de búsqueda ayuda a reducir el riesgo, aunque no lo elimina por completo.

## 8. Conclusión
La modificación de mayor impacto debe identificarse a partir de los resultados ejecutados. El modelo optimizado solo se recomienda si la mejora en F1-score o ROC-AUC es suficientemente clara para justificar el tiempo adicional de GridSearchCV. Si la diferencia es mínima, el modelo reducido puede ser preferible porque utiliza menos variables y conserva una interpretación más sencilla.

La principal limitación es que se utiliza un solo dataset y un espacio de hiperparámetros acotado. Como siguiente paso se podría probar otra técnica de selección, ampliar la búsqueda o realizar una validación externa para comprobar que la mejora sea estable.